In [17]:
df=spark.read.parquet("abfss://bronze@idestorageaccount.dfs.core.windows.net/nyc_yellow_taxi")
df.printSchema()
print(df.count())
display(df.limit(5))

StatementMeta(idesparkpool, 0, 18, Finished, Available, Finished, False)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)

3582628


SynapseWidget(Synapse.DataFrame, 3a904083-3f71-4bbb-b33c-e0fd343b1b89)

In [12]:
from pyspark.sql import functions as F

# nulls per column
display(df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]))

# negative/zero fares, distances
print("neg_fares")
print(df.filter((F.col("fare_amount") <= 0) | (F.col("trip_distance") <= 0)).count())

# passenger count sanity check
display(df.select("passenger_count").distinct().orderBy("passenger_count"))

# datetime range check
display(df.select(F.min("tpep_pickup_datetime"), F.max("tpep_pickup_datetime")))

# dropoff before pickup (impossible trips)
print(f"dropoff before pickup: {df.filter(F.col('tpep_dropoff_datetime') < F.col('tpep_pickup_datetime')).count()}")
# duplicate rows
print(df.count() - df.dropDuplicates().count())

## <mark>Silver Rules:</mark>
#### **1. null values:** 
passenger_count,RatecodeID,store_and_fwd_flag,congestion_surcharge,Airport_fee - each of these cols hold 426,190 null values (~11.9% of the data)<br>
treatment: impute values<br>
passenger_count- 1 (most common val)<br>
RatecodeID-99(as per TLC dict, unknown or null rate)<br> 
store_and_fwd_flag-'N'(majority)<br>
congestion_surcharge-0<br>
Airport_fee-0<br>
#### **2. tpep_pickup_datetime range**
Min = 2002-12-31 (garbage), max = 2024-04-01 (TLC files always spill a few hours into next month)
treatment: Filter to valid expected month range only 
#### **3. dropoff before pickup**
117 rows
treatment: Drop, physically invalid trips
#### **4. duplicates**
0 which implies no treatment
#### **5. fare_amount<0 or trip_distance<=0:**
142563 rows => Drop them off 
#### **6. passenger count<0:**
present in distinct values (separate from null)
treatment: keep (TLC legitimately inc dispatch or unmetered 0 passenger flag rows )

In [16]:
''' Silver Transformation Code '''

silver_df = (
    df
    .fillna({
        "passenger_count": 1,
        "RatecodeID": 99,
        "store_and_fwd_flag": "N",
        "congestion_surcharge": 0.0,
        "Airport_fee": 0.0
    })
    .filter(
        (F.col("tpep_pickup_datetime") >= "2024-03-01") &
        (F.col("tpep_pickup_datetime") < "2024-04-01")
    )
    .filter(F.col("tpep_dropoff_datetime") >= F.col("tpep_pickup_datetime"))
    .filter((F.col("fare_amount") > 0) & (F.col("trip_distance") > 0))
    # Add trip_duration_minutes for downstream Gold use
    .withColumn(
        "trip_duration_minutes",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
    )
)

print(f"Records Lost Post Clean Up: {df.count()-silver_df.count()}")  # sanity check vs expected ~3.44M after drops

silver_df.write.format("delta").mode("overwrite").save(
    "abfss://silver@idestorageaccount.dfs.core.windows.net/nyc_yellow_taxi"
)